<a href="https://colab.research.google.com/github/RothRothRoth/DeepLearning_CNN/blob/main/Copy_of_Road_Sign_Recognition_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Road Sign Recognition Using Deep Learning

## Final Project

**Task:** Image Classification  
**Dataset:** Cambodia Traffic Signs Dataset (CamTSD)  
**Number of Classes:** 26  
**Approaches:**
1. CNN trained from scratch
2. ResNet-18 with frozen pretrained backbone
3. ResNet-18 with full fine-tuning

### Objective
Develop and compare deep learning approaches for recognizing Cambodian traffic signs from images.

### Evaluation
The approaches will be evaluated using the same held-out test set and appropriate classification metrics, including accuracy, precision, recall, F1-score, and confusion matrices.

In [ ]:
# ============================================
# Cell 2: Environment & Reproducibility
# ============================================

import os
import random
import numpy as np
import torch

# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch version:", torch.__version__)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU: Not available")

PyTorch version: 2.11.0+cu128
Device: cuda
GPU: Tesla T4


In [ ]:
# ============================================
# Cell 3: Google Drive & Project Paths
# ============================================

from google.colab import drive

# Mount Google Drive
drive.mount("/content/drive", force_remount=True)

# Main project directory
PROJECT_DIR = "/content/drive/MyDrive/Road_Sign_Recognition_CNN"

# Project subdirectories
DATASET_DIR = os.path.join(PROJECT_DIR, "dataset")
RAW_DIR = os.path.join(DATASET_DIR, "raw")
PROCESSED_DIR = os.path.join(DATASET_DIR, "processed")
TEST_DIR = os.path.join(DATASET_DIR, "test")

MODELS_DIR = os.path.join(PROJECT_DIR, "models")
RESULTS_DIR = os.path.join(PROJECT_DIR, "results")

# Create required folders if they don't exist
for folder in [
    PROJECT_DIR,
    DATASET_DIR,
    RAW_DIR,
    PROCESSED_DIR,
    TEST_DIR,
    MODELS_DIR,
    RESULTS_DIR
]:
    os.makedirs(folder, exist_ok=True)

print("Project directory:")
print(PROJECT_DIR)

print("\nProject folders:")
for folder in [
    DATASET_DIR,
    RAW_DIR,
    PROCESSED_DIR,
    TEST_DIR,
    MODELS_DIR,
    RESULTS_DIR
]:
    print("✓", folder)

Mounted at /content/drive
Project directory:
/content/drive/MyDrive/Road_Sign_Recognition_CNN

Project folders:
✓ /content/drive/MyDrive/Road_Sign_Recognition_CNN/dataset
✓ /content/drive/MyDrive/Road_Sign_Recognition_CNN/dataset/raw
✓ /content/drive/MyDrive/Road_Sign_Recognition_CNN/dataset/processed
✓ /content/drive/MyDrive/Road_Sign_Recognition_CNN/dataset/test
✓ /content/drive/MyDrive/Road_Sign_Recognition_CNN/models
✓ /content/drive/MyDrive/Road_Sign_Recognition_CNN/results


In [ ]:
# ============================================
# Cell 4: Dataset Verification
# ============================================

import os

# Check processed dataset
print("Processed dataset exists:", os.path.exists(PROCESSED_DIR))

# Get class folders
class_folders = sorted([
    folder for folder in os.listdir(PROCESSED_DIR)
    if os.path.isdir(os.path.join(PROCESSED_DIR, folder))
])

print("\nNumber of classes:", len(class_folders))

print("\nClasses:")
for i, class_name in enumerate(class_folders, start=1):
    print(f"{i:02d}. {class_name}")

# Count images in each class
print("\nImages per class:")

total_images = 0

for class_name in class_folders:
    class_path = os.path.join(PROCESSED_DIR, class_name)

    image_count = len([
        file for file in os.listdir(class_path)
        if file.lower().endswith((".jpg", ".jpeg", ".png"))
    ])

    total_images += image_count

    print(f"{class_name:30s} : {image_count}")

print("\n--------------------------------")
print("Total images:", total_images)
print("Total classes:", len(class_folders))
print("--------------------------------")

Processed dataset exists: True

Number of classes: 26

Classes:
01. 30_SPEED_LIMIT
02. 40_SPEED_LIMIT
03. 60_SPEED_LIMIT
04. 80_SPEED_LIMIT
05. CARRIAGE_WAY_NARROWS
06. CHILDREN_CROSSING
07. CROSS_ROAD
08. DIRECTION
09. END_SPEED_LIMIT
10. HOSPITAL
11. KEEP_RIGHT
12. KM_POST
13. LEFT_BEND
14. NO_ENTRY
15. NO_HORN
16. NO_PARKING
17. NO_UTURN
18. PEDESTRAIN_CROSSING
19. PEDESTRAIN_CR_AREA
20. PRIORITY_ROAD
21. RIGHT_BEND
22. ROAD_JUNCTION_ON_THE_LEFT
23. ROAD_JUNCTION_ON_THE_RIGHT
24. SLOW_DOWN
25. STAGGERED_JUNCTION_RL
26. U_TURN

Images per class:
30_SPEED_LIMIT                 : 30
40_SPEED_LIMIT                 : 94
60_SPEED_LIMIT                 : 48
80_SPEED_LIMIT                 : 37
CARRIAGE_WAY_NARROWS           : 29
CHILDREN_CROSSING              : 188
CROSS_ROAD                     : 42
DIRECTION                      : 152
END_SPEED_LIMIT                : 52
HOSPITAL                       : 53
KEEP_RIGHT                     : 327
KM_POST                        : 738
LEFT_BEND 

In [ ]:
# ============================================
# Cell 5: Locate Original Dataset
# ============================================

KAGGLE_DIR = "/kaggle/input/cambodia-traffic-signs-dataset"

print("Dataset exists:", os.path.exists(KAGGLE_DIR))

if os.path.exists(KAGGLE_DIR):
    print("\nDataset contents:")
    for item in sorted(os.listdir(KAGGLE_DIR)):
        print("✓", item)

    IMAGE_DIR = os.path.join(KAGGLE_DIR, "images")
    DATA_DIR = os.path.join(KAGGLE_DIR, "data")
    JSON_PATH = os.path.join(DATA_DIR, "CAM_TSR_v3_json.json")

    print("\nImages folder exists:", os.path.exists(IMAGE_DIR))
    print("Data folder exists:", os.path.exists(DATA_DIR))
    print("Annotation JSON exists:", os.path.exists(JSON_PATH))
else:
    print("\n⚠️ Original Kaggle dataset is not available in this Colab runtime.")

Dataset exists: False

⚠️ Original Kaggle dataset is not available in this Colab runtime.


In [ ]:
# ============================================
# Cell 6: Check Raw Dataset
# ============================================

print("Raw dataset folder:")
print(RAW_DIR)

print("\nContents:")

if os.path.exists(RAW_DIR):
    items = os.listdir(RAW_DIR)

    if len(items) == 0:
        print("⚠️ Raw folder is empty.")
    else:
        for item in sorted(items):
            print("✓", item)
else:
    print("⚠️ Raw folder does not exist.")

Raw dataset folder:
/content/drive/MyDrive/Road_Sign_Recognition_CNN/dataset/raw

Contents:
✓ archive.zip
✓ data
✓ images
✓ labels
✓ via.html


In [ ]:
# ============================================
# Cell 8: Extract Original Dataset
# ============================================

import zipfile
import os
import shutil

if len(zip_files) == 0:
    raise FileNotFoundError("No ZIP file found in the raw dataset folder.")

ZIP_PATH = zip_files[0]

# Extract to fast Colab local storage
LOCAL_RAW_DIR = "/content/road_sign_dataset"

print("Extracting:")
print(ZIP_PATH)

# Remove old incomplete extraction if it exists
if os.path.exists(LOCAL_RAW_DIR):
    shutil.rmtree(LOCAL_RAW_DIR)

os.makedirs(LOCAL_RAW_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(LOCAL_RAW_DIR)

print("\n✅ Extraction complete!")

print("\nDataset contents:")
for item in sorted(os.listdir(LOCAL_RAW_DIR)):
    print("📁", item)

Extracting:
/content/drive/MyDrive/Road_Sign_Recognition_CNN/dataset/raw/archive.zip

✅ Extraction complete!

Dataset contents:
📁 data
📁 images
📁 labels
📁 via.html


In [ ]:
# ============================================
# Cell 9: Verify Original Dataset
# ============================================

IMAGE_DIR = os.path.join(RAW_DIR, "images")
LABEL_DIR = os.path.join(RAW_DIR, "labels")
ANNOTATION_DIR = os.path.join(RAW_DIR, "data")

# Find annotation JSON files
json_files = [
    f for f in os.listdir(ANNOTATION_DIR)
    if f.lower().endswith(".json")
]

# Count images
image_files = [
    f for f in os.listdir(IMAGE_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

# Count label files
label_files = [
    f for f in os.listdir(LABEL_DIR)
    if f.lower().endswith(".txt")
]

print("========== ORIGINAL DATASET ==========")
print("Images:", len(image_files))
print("YOLO label files:", len(label_files))
print("JSON annotation files:", len(json_files))

print("\nJSON files:")
for f in json_files:
    print("✓", f)

print("\nImage folder:", IMAGE_DIR)
print("Label folder:", LABEL_DIR)
print("Annotation folder:", ANNOTATION_DIR)

========== ORIGINAL DATASET ==========
Images: 2766
YOLO label files: 2757
JSON annotation files: 8

JSON files:
✓ CAM_TSR_v2.json
✓ CAM_TSR_json.json
✓ CAM_TSR_v2_json.json
✓ CAM_TSR_v3_json_state.json
✓ CAM_TSR_v3_json.json
✓ CAM_TSR.json
✓ CAM_TSR_attributes.json
✓ CAM_TSR_v3.json

Image folder: /content/drive/MyDrive/Road_Sign_Recognition_CNN/dataset/raw/images
Label folder: /content/drive/MyDrive/Road_Sign_Recognition_CNN/dataset/raw/labels
Annotation folder: /content/drive/MyDrive/Road_Sign_Recognition_CNN/dataset/raw/data


In [ ]:
# ============================================
# Cell 10: Inspect Traffic Sign Annotations
# ============================================

import json

JSON_PATH = os.path.join(
    ANNOTATION_DIR,
    "CAM_TSR_v3_json.json"
)

with open(JSON_PATH, "r", encoding="utf-8") as f:
    annotations = json.load(f)

print("Annotation file loaded successfully!")
print("Type:", type(annotations))

if isinstance(annotations, dict):
    print("Number of records:", len(annotations))
    print("\nFirst record:")
    first_key = next(iter(annotations))
    print("Key:", first_key)
    print("Value:", annotations[first_key])
else:
    print("Number of records:", len(annotations))

Annotation file loaded successfully!
Type: <class 'dict'>
Number of records: 2757

First record:
Key: GX010109.MP4_snapshot_00.10.234.jpg1255446
Value: {'filename': 'GX010109.MP4_snapshot_00.10.234.jpg', 'size': 579503, 'regions': [{'shape_attributes': {'name': 'rect', 'x': 2744, 'y': 655, 'width': 162, 'height': 156}, 'region_attributes': {'Object': 'Traffic Sign', 'Class': 'Warning', 'Name': 'CHILDREN CROSSING', 'Status': 'Good'}}, {'shape_attributes': {'name': 'rect', 'x': 2754, 'y': 807, 'width': 140, 'height': 132}, 'region_attributes': {'Object': 'Traffic Sign', 'Class': 'Prohibitory', 'Name': '40 SPEED LIMIT', 'Status': 'Good'}}], 'file_attributes': {}}


In [ ]:
# Check annotation statistics

total_regions = 0
undefined_count = 0
images_with_multiple_signs = 0

for record in annotations.values():
    regions = record.get("regions", [])

    total_regions += len(regions)

    if len(regions) > 1:
        images_with_multiple_signs += 1

    for region in regions:
        attrs = region.get("region_attributes", {})
        if attrs.get("Name") == "UNDEFINED":
            undefined_count += 1

print("Total traffic-sign annotations:", total_regions)
print("Images containing multiple signs:", images_with_multiple_signs)
print("UNDEFINED annotations:", undefined_count)

Total traffic-sign annotations: 3490
Images containing multiple signs: 627
UNDEFINED annotations: 117


In [ ]:
# Create a stable class list

class_names = sorted(name_counts.keys())

print("Number of classes:", len(class_names))
print("\nClass mapping:")

for i, name in enumerate(class_names):
    print(f"{i:2d} -> {name}")

Number of classes: 49

Class mapping:
 0 -> 30 SPEED LIMIT
 1 -> 40 SPEED LIMIT
 2 -> 60 SPEED LIMIT
 3 -> 80 SPEED LIMIT
 4 -> ANIMAL CROSSING
 5 -> BUMPY ROAD
 6 -> CARRIAGE WAY NARROWS
 7 -> CHILDREN CROSSING
 8 -> CROSS ROAD
 9 -> DIRECTION
10 -> DOUBLE BEND
11 -> END PROHIBIT
12 -> END SPEED LIMIT
13 -> GIVE WAY AT RB
14 -> HEIGHT LIMIT
15 -> HOSPITAL
16 -> KEEP RIGHT
17 -> KM POST
18 -> LEFT BEND
19 -> MERG LANE
20 -> NO ENTRY
21 -> NO ENTRY TRUCK
22 -> NO HORN
23 -> NO OVERTAKING
24 -> NO PARKING
25 -> NO STOPPING
26 -> NO UTURN
27 -> PARKING
28 -> PEDESTRAIN CR AREA
29 -> PEDESTRAIN CROSSING
30 -> PRIORITY ROAD
31 -> RAILWAY CROSSING
32 -> RIGHT BEND
33 -> ROAD HUMP
34 -> ROAD JUNCTION ON THE LEFT
35 -> ROAD JUNCTION ON THE RIGHT
36 -> ROAD WORK AHEAD
37 -> ROUND ABOUT
38 -> SLOW DOWN
39 -> STAGGERED JUNCTION, LR
40 -> STAGGERED JUNCTION, RL
41 -> STOP
42 -> TRAFFIC LIGHTS
43 -> TURN RIGHT
44 -> U TURN
45 -> UNDEFINED
46 -> WINDING ROAD
47 -> Y JUNCTION LEFT
48 -> Y JUNCTION RI

In [ ]:
!git clone https://github.com/RothRothRoth/DeepLearning_CNN.git /content/DeepLearning_CNN




Cloning into '/content/DeepLearning_CNN'...
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 9 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (9/9), 602.82 KiB | 10.76 MiB/s, done.
Resolving deltas: 100% (2/2), done.


In [ ]:
!ls -lh /content/drive/MyDrive/Road_Sign_Recognition_CNN/notebooks/

total 0
